# 1. Title and Overview

This notebook implements the archive V1 LLM-as-a-judge workflow for compliance reasoning evaluation.

What this notebook does:
- loads rules, calibration, rubrics, and model output tables;
- runs metric-level judging (Stage 1) with one judge call per metric;
- computes deterministic composite metrics locally (Stage 1b);
- generates overall assessment from fixed metric values (Stage 2);
- exports final CSV/JSONL artifacts.


## 2. Configuration

This section defines paths, judge settings, execution mode, and output file names.
Use `DRY_RUN=True` for structure checks without external API calls; use `DRY_RUN=False` for real OpenRouter judging.


In [85]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = None
for candidate in [Path.cwd() / "eval.env", Path.cwd().parent / "eval.env", Path.cwd().parent.parent / "eval.env", Path.cwd().parent.parent.parent / "eval.env"]:
    if candidate.exists():
        env_path = candidate
        break

if env_path is not None:
    load_dotenv(env_path)
else:
    load_dotenv()

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
assert OPENROUTER_API_KEY, "Set OPENROUTER_API_KEY in eval.env"
print("OPENROUTER_API_KEY is set")

OPENROUTER_API_KEY is set


In [86]:
from pathlib import Path

# OpenRouter configuration
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY_ENV = "OPENROUTER_API_KEY"
JUDGE_MODEL = "anthropic/claude-sonnet-4.5"

# OpenRouter key bootstrap (env and notebook variable are both supported)
OPENROUTER_API_KEY = str(globals().get("OPENROUTER_API_KEY", "") or "").strip()
if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = os.getenv(OPENROUTER_API_KEY_ENV, "").strip()
if OPENROUTER_API_KEY:
    os.environ[OPENROUTER_API_KEY_ENV] = OPENROUTER_API_KEY

OPENROUTER_HTTP_REFERER = os.getenv("OPENROUTER_HTTP_REFERER", "").strip()
OPENROUTER_X_TITLE = os.getenv("OPENROUTER_X_TITLE", "LLM-as-a-judge").strip()

# Execution controls
DRY_RUN = True           # False -> real OpenRouter calls
MAX_ITEMS = None         # e.g. 20 for quick debug
TEMPERATURE = 0.0
MAX_TOKENS_STAGE1 = 700
MAX_TOKENS_STAGE2 = 250
RETRY_MAX = 6

# Retrieval controls
MIN_LOCAL_CALIB = 4
MAX_CALIB = 5
MAX_LEVEL3_ADDS = 4
SIM_WEIGHT = 0.80
AGREEMENT_WEIGHT = 0.20

# Embedding model
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Input tables (real project files)
ROOT = Path("..")
RULES_CSV = ROOT / "20_03_VQA_Rules_Scenes_Templates(rules_sort).csv"
RULE_FIGURE_CSV = ROOT / "09_01_VQA_Rules_Scenes_Templates(rule_figure).csv"
CALIBRATION_CSV = ROOT / "03_04_reasonLLMjudgeCalibr.csv"
RUBRICS_CSV = ROOT / "03_04_VQA_metric_rubrics.csv"

EVAL_DIR = ROOT / "notebooks"
EVAL_FILES = [
    EVAL_DIR / "3_2_to_3_5_clean_claude-opus-4.6_V2.csv",
    EVAL_DIR / "3_2_to_3_5_clean_gemini_3_pro_prev_V2.csv",
    EVAL_DIR / "3_2_to_3_5_clean_gpt-5_2_V2.csv",
]

# Stage outputs
OUT_DIR = ROOT / "results"
STAGE1_METRIC_CSV = OUT_DIR / "llm_judge_stage1_metric_level.csv"
STAGE1_RAW_JSONL = OUT_DIR / "llm_judge_stage1_raw.jsonl"
STAGE1_MERGED_CSV = OUT_DIR / "llm_judge_stage1_merged.csv"
STAGE1B_COMPOSITE_CSV = OUT_DIR / "llm_judge_stage1b_composites.csv"
STAGE2_OVERALL_CSV = OUT_DIR / "llm_judge_stage2_overall.csv"
FINAL_MERGED_CSV = OUT_DIR / "llm_judge_final_merged.csv"

# Metric coverage
# These are metric-level judge calls (one call = one metric).
JUDGE_METRICS = [
    "Hallucination",
    "Redundancy",
    "Commonsense",
    "Completeness",
    "CRCS - Entity Grounding",
    "CRCS - Visual Alignment",
    "CRCS - Reference Mention Accuracy",
    "Reasoning Structure Fidelity (RSF)"
]

# These are computed deterministically in Stage 1b (no additional judge call required).
DETERMINISTIC_METRICS = [
    "Reasoning Structure Fidelity (RSF)",
    "CRCS - Final Verdict Accuracy",
    "Cross-Representational Coherence Score (CRCS)",
]

print("Configuration loaded")
print("DRY_RUN:", DRY_RUN)
print("Judge model:", JUDGE_MODEL)
print("Default JUDGE_METRICS:", JUDGE_METRICS)


Configuration loaded
DRY_RUN: True
Judge model: anthropic/claude-sonnet-4.5
Default JUDGE_METRICS: ['Hallucination', 'Redundancy', 'Commonsense', 'Completeness', 'CRCS - Entity Grounding', 'CRCS - Visual Alignment', 'CRCS - Reference Mention Accuracy']


## 3. Imports

Imports for data processing, retrieval, and API calls.
The notebook also defines a cosine-similarity helper used in retrieval ranking.


In [87]:
import os
import re
import json
import time
import math
import hashlib
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import requests

try:
    from sentence_transformers import SentenceTransformer
except Exception as e:
    raise ImportError(
        "sentence-transformers is required. Install with: pip install sentence-transformers"
    ) from e


def cosine_sim_1_to_many(query_vec: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    q = np.asarray(query_vec, dtype=float)
    m = np.asarray(matrix, dtype=float)
    qn = np.linalg.norm(q)
    mn = np.linalg.norm(m, axis=1)
    denom = np.maximum(mn * qn, 1e-12)
    return (m @ q) / denom


## 4. Input Schema Validation

This block validates required columns for each logical table (`rules_table`, `calibration_examples`, `metric_rubrics`, `evaluation_items`).
Execution should stop here if any required schema field is missing.


In [88]:
def normalize_colname(name: str) -> str:
    s = str(name).strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s


def drop_unnamed(df: pd.DataFrame) -> pd.DataFrame:
    keep = []
    for c in df.columns:
        cs = str(c).strip()
        if cs == "":
            continue
        if cs.lower().startswith("unnamed"):
            continue
        keep.append(c)
    return df[keep].copy()


def read_csv_flexible(path: Path, sep: str) -> pd.DataFrame:
    last = None
    for enc in ("utf-8", "utf-8-sig", "cp1251", "latin1"):
        try:
            return pd.read_csv(path, sep=sep, encoding=enc, engine="python")
        except Exception as e:
            last = e
    raise RuntimeError(f"Failed to read {path}: {last}")


def canonicalize_columns(df: pd.DataFrame, aliases: Dict[str, List[str]]) -> pd.DataFrame:
    df = df.copy()
    norm_map = {normalize_colname(c): c for c in df.columns}
    rename = {}
    for canonical, cand in aliases.items():
        for a in cand:
            key = normalize_colname(a)
            if key in norm_map:
                rename[norm_map[key]] = canonical
                break
    if rename:
        df = df.rename(columns=rename)
    return df


def validate_required_columns(df: pd.DataFrame, required: List[str], table_name: str):
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{table_name}: missing required columns -> {missing}")


def optional_column_report(df: pd.DataFrame, optional: List[str], table_name: str):
    missing_opt = [c for c in optional if c not in df.columns]
    print(f"{table_name}: optional missing -> {missing_opt}")


## 5. Data Loading

This section loads source CSVs and normalizes column names through alias mappings.
Multiple model output files are concatenated into one `evaluation_items` table with `candidate_model` and `source_eval_file` tags.


In [89]:
# Column aliases per table (resilient to capitalization / naming variants)
RULES_ALIASES = {
    "parent_rule_id": ["parent_rule_id", "ParentRuleID", "classification_parent_rule_id"],
    "parent_rule_text": ["parent_rule_text", "ParentRuleText"],
    "rule_id": ["rule_id", "RuleID"],
    "rule_text_atomic": ["rule_text_atomic", "rule_text", "RuleTextAtomic"],
    "rule_figure_caption": ["rule_figure_caption", "caption", "figure_caption"],
    "ambiguity_remark": ["ambiguity_remark", "Ambiguity_remark", "ambiguity_note"],
    "counter_notes": ["counter_notes", "counter_note"],
    "figure_id": ["figure_id", "rule_figure_id"],
}

CALIB_ALIASES = {
    "reason_gt_id": ["reason_gt_id"],
    "parent_rule_id": ["parent_rule_id", "ParentRuleID"],
    "question_id": ["question_id"],
    "scene_id": ["scene_id"],
    "task_family": ["task_family"],
    "ground_truth_answer": ["ground_truth_answer"],
    "answer_to_evaluate": ["answer_to_evaluate", "candidate_reasoning_chain"],
    "survey_question_id": ["survey_question_id"],
    "human_verdict_distribution": ["human_verdict_distribution"],
    "human_verdict_consensus": ["human_verdict_consensus"],
    "human_verdict_agreement": ["human_verdict_agreement"],
    "human_completeness_mean": ["human_completeness_mean"],
    "human_completeness_std": ["human_completeness_std"],
    "human_completeness_agreement": ["human_completeness_agreement", "human_completeness_collapsed_agreement"],
    "human_logical_mean": ["human_logical_mean"],
    "human_logical_std": ["human_logical_std"],
    "human_redundancy_mean": ["human_redundancy_mean"],
    "human_redundancy_std": ["human_redundancy_std"],
    "counter_note": ["counter_note"],
    "rule_interpretation_note": ["rule_interpretation_note"],
    "reasoning_pattern": ["reasoning_pattern"],
    "error_type": ["error_type"],
    "ambiguity_level": ["ambiguity_level"],
    "figure_id": ["figure_id", "rule_figure_id"],
    "survey_stats": ["survey_stats"],
    # Calibration split can be named either "split" or "set".
    "split": ["split", "set", "Split", "Set", "dataset_split", "subset"],
}

RUBRIC_ALIASES = {
    "metric_name": ["metric_name"],
    "definition": ["definition"],
    "score_scale": ["score_scale"],
    "scoring_guidelines": ["scoring_guidelines"],
    "anti_bias_note": ["anti_bias_note"],
}

EVAL_ALIASES = {
    "generated_question_id": ["generated_question_id"],
    "template_id": ["template_id"],
    "question_text": ["question_text"],
    "scene_id": ["scene_id"],
    "file_path": ["file_path"],
    "figure_path": ["figure_path"],
    "RuleID": ["RuleID", "rule_id"],
    "ParentRuleID": ["ParentRuleID", "parent_rule_id"],
    "Classification": ["Classification"],
    "Ambiguity": ["Ambiguity"],
    "ClassificationParent": ["ClassificationParent", "classification_parent"],
    "rule_figure_id": ["rule_figure_id", "figure_id"],
    "correct_answer": ["correct_answer"],
    "CorrectAnswer": ["CorrectAnswer"],
    "model_answer": ["model_answer"],
    "openrouter_request_id": ["openrouter_request_id"],
    "error_reason": ["error_reason"],
    "row_index": ["row_index"],
    "bool_model_answer": ["bool_model_answer"],
}

# Load raw tables
rules_raw = drop_unnamed(read_csv_flexible(RULES_CSV, sep=';'))
rule_fig_raw = drop_unnamed(read_csv_flexible(RULE_FIGURE_CSV, sep=';'))
calib_raw = drop_unnamed(read_csv_flexible(CALIBRATION_CSV, sep=';'))
rubric_raw = drop_unnamed(read_csv_flexible(RUBRICS_CSV, sep=';'))

rules_table = canonicalize_columns(rules_raw, RULES_ALIASES)
rule_fig_table = canonicalize_columns(rule_fig_raw, {"figure_id":["figure_id"], "rule_figure_caption":["caption","rule_figure_caption"]})
calibration_examples = canonicalize_columns(calib_raw, CALIB_ALIASES)
metric_rubrics = canonicalize_columns(rubric_raw, RUBRIC_ALIASES)

# Merge caption into rules table when missing
if "rule_figure_caption" not in rules_table.columns:
    rules_table["rule_figure_caption"] = ""

if "figure_id" in rules_table.columns and "figure_id" in rule_fig_table.columns:
    tmp = rule_fig_table[["figure_id", "rule_figure_caption"]].copy()
    rules_table = rules_table.merge(tmp, on="figure_id", how="left", suffixes=("", "_fig"))
    rules_table["rule_figure_caption"] = rules_table["rule_figure_caption"].fillna("")
    if "rule_figure_caption_fig" in rules_table.columns:
        rules_table["rule_figure_caption"] = rules_table["rule_figure_caption"].mask(
            rules_table["rule_figure_caption"].astype(str).str.strip()=="",
            rules_table["rule_figure_caption_fig"].fillna("")
        )
        rules_table = rules_table.drop(columns=["rule_figure_caption_fig"])

# Load evaluation items from all model outputs and add candidate_model
_eval_frames = []
missing_eval_files = [str(fp) for fp in EVAL_FILES if not Path(fp).exists()]
if missing_eval_files:
    raise FileNotFoundError(
        "Missing EVAL_FILES:\n- " + "\n- ".join(missing_eval_files)
    )

for fp in EVAL_FILES:
    d = drop_unnamed(pd.read_csv(fp))
    d = canonicalize_columns(d, EVAL_ALIASES)
    m = re.search(r"clean_(.+?)\.csv$", fp.name)
    d["candidate_model"] = m.group(1) if m else fp.stem
    d["source_eval_file"] = fp.name
    _eval_frames.append(d)

evaluation_items = pd.concat(_eval_frames, ignore_index=True)

if MAX_ITEMS is not None:
    evaluation_items = evaluation_items.head(MAX_ITEMS).copy()

print("Loaded shapes:")
print("rules_table:", rules_table.shape)
print("calibration_examples:", calibration_examples.shape)
print("metric_rubrics:", metric_rubrics.shape)
print("evaluation_items:", evaluation_items.shape)

# Sync metric coverage with the rubric table:
# - JUDGE_METRICS are all rubric metrics except deterministic ones.
def metric_key_boot(s: Any) -> str:
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    t = str(s).strip().lower()
    if t in {"", "nan", "none", "null", "-"}:
        return ""
    for ch in ["-", "–", "—", "−", "‑", "?"]:
        t = t.replace(ch, "-")
    t = re.sub(r"[^a-z0-9]+", " ", t)
    return re.sub(r"\s+", " ", t).strip()

rubric_metric_names = []
for m in metric_rubrics.get("metric_name", pd.Series(dtype=object)).tolist():
    if m is None or (isinstance(m, float) and pd.isna(m)):
        continue
    txt = str(m).strip()
    if txt and txt.lower() not in {"nan", "none", "null", "-"}:
        rubric_metric_names.append(txt)
det_keys = {metric_key_boot(m) for m in DETERMINISTIC_METRICS}
judge_from_rubric = [m for m in rubric_metric_names if metric_key_boot(m) not in det_keys]
if judge_from_rubric:
    JUDGE_METRICS = list(dict.fromkeys(judge_from_rubric))

covered_keys = {metric_key_boot(m) for m in JUDGE_METRICS} | det_keys
uncovered_rubric_metrics = [m for m in rubric_metric_names if metric_key_boot(m) not in covered_keys]

print("JUDGE_METRICS (from rubric, non-deterministic):", JUDGE_METRICS)
print("DETERMINISTIC_METRICS:", DETERMINISTIC_METRICS)
if uncovered_rubric_metrics:
    print("WARNING: rubric metrics not covered:", uncovered_rubric_metrics)


Loaded shapes:
rules_table: (69, 17)
calibration_examples: (29, 16)
metric_rubrics: (10, 6)
evaluation_items: (420, 21)
JUDGE_METRICS (from rubric, non-deterministic): ['Hallucination', 'Redundancy', 'Commonsense', 'Completeness', 'CRCS – Reference Mention Accuracy', 'CRCS – Entity Grounding', 'CRCS – Visual Alignment']
DETERMINISTIC_METRICS: ['Reasoning Structure Fidelity (RSF)', 'CRCS - Final Verdict Accuracy', 'Cross-Representational Coherence Score (CRCS)']


## 6. Data Normalization

ID, text, and category fields are normalized to reduce join/lookup errors.
The output of this stage is a schema-consistent set of tables used by retrieval and judging.


In [90]:
def normalize_id(v: Any) -> str:
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    s = str(v).strip().strip('"').strip("'")
    if s == "" or s.lower() in {"nan", "none", "null", "-"}:
        return ""
    try:
        f = float(s.replace(",", "."))
        if f.is_integer():
            return str(int(f))
    except Exception:
        pass
    return s


def clean_text(v: Any) -> str:
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    s = str(v).strip()
    if s.lower() in {"nan", "none", "null", "-"}:
        return ""
    return s


def parse_float(v: Any) -> Optional[float]:
    s = clean_text(v)
    if s == "":
        return None
    s = s.replace(",", ".")
    try:
        return float(s)
    except Exception:
        return None


def parse_survey_blob(blob: Any) -> Dict[str, str]:
    txt = clean_text(blob)
    if txt == "":
        return {}
    out = {}
    for raw in txt.splitlines():
        line = raw.strip()
        if not line or ":" not in line:
            continue
        k, v = line.split(":", 1)
        key = normalize_colname(k)
        out[key] = v.strip()
    return out


def normalize_split_label(v: Any) -> str:
    s = clean_text(v).lower().replace(" ", "_").replace("-", "_")
    alias = {
        "anchor": "anchor",
        "hardvalidation": "hard_validation",
        "hard_validation": "hard_validation",
        "validation_hard": "hard_validation",
        "holdout": "holdout",
        "test": "holdout",
    }
    s = alias.get(s, s)
    if s in {"anchor", "hard_validation", "holdout"}:
        return s
    return s

# Normalize key IDs
for col in ["parent_rule_id", "rule_id", "figure_id"]:
    if col in rules_table.columns:
        rules_table[col] = rules_table[col].apply(normalize_id)

for col in ["reason_gt_id", "parent_rule_id", "question_id", "scene_id", "figure_id"]:
    if col in calibration_examples.columns:
        calibration_examples[col] = calibration_examples[col].apply(normalize_id)

for col in ["generated_question_id", "scene_id", "RuleID", "ParentRuleID", "rule_figure_id", "template_id", "row_index"]:
    if col in evaluation_items.columns:
        evaluation_items[col] = evaluation_items[col].apply(normalize_id)

# Parse survey_stats blob into normalized fields when needed
survey_required = [
    "survey_question_id",
    "human_verdict_distribution",
    "human_verdict_consensus",
    "human_verdict_agreement",
    "human_completeness_mean",
    "human_completeness_std",
    "human_completeness_agreement",
    "human_logical_mean",
    "human_logical_std",
    "human_redundancy_mean",
    "human_redundancy_std",
]

if "survey_stats" in calibration_examples.columns:
    parsed = calibration_examples["survey_stats"].apply(parse_survey_blob)
    for c in survey_required:
        if c not in calibration_examples.columns:
            calibration_examples[c] = parsed.apply(lambda d: d.get(c, ""))
        else:
            calibration_examples[c] = calibration_examples[c].mask(
                calibration_examples[c].astype(str).str.strip().isin(["", "nan", "None", "-"]),
                parsed.apply(lambda d: d.get(c, ""))
            )

# Numeric survey conversions
for c in [
    "human_verdict_agreement",
    "human_completeness_mean",
    "human_completeness_std",
    "human_completeness_agreement",
    "human_logical_mean",
    "human_logical_std",
    "human_redundancy_mean",
    "human_redundancy_std",
]:
    if c in calibration_examples.columns:
        calibration_examples[c] = calibration_examples[c].apply(parse_float)

# Text cleanup
for df in (rules_table, calibration_examples, metric_rubrics, evaluation_items):
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = df[c].apply(clean_text)

# Split normalization and validation (hard methodological constraint)
if "split" not in calibration_examples.columns:
    raise ValueError(
        "calibration_examples is missing split/set column. "
        "Expected one of: split, set, dataset_split, subset"
    )

calibration_examples["split"] = calibration_examples["split"].apply(normalize_split_label)

known_splits = {"anchor", "hard_validation", "holdout"}
unknown_splits = sorted({s for s in calibration_examples["split"].unique().tolist() if s and s not in known_splits})
if unknown_splits:
    raise ValueError(f"Unknown calibration split labels found: {unknown_splits}")

# Likert folding helpers
# mean_likert = arithmetic mean of 1-5 ratings
# std_likert = sample std -> pandas.std(ddof=1)

def fold_likert_completeness_or_logical(x: Any) -> str:
    x_num = parse_float(x)
    if x_num is None or (isinstance(x_num, float) and pd.isna(x_num)):
        return ""
    if x_num <= 2:
        return "low"
    if x_num < 4:
        return "partial"
    return "high"


def fold_likert_redundancy_reverse(x: Any) -> str:
    x_num = parse_float(x)
    if x_num is None or (isinstance(x_num, float) and pd.isna(x_num)):
        return ""
    if x_num <= 2:
        return "low_redundancy"
    if x_num < 4:
        return "partial"
    return "high_redundancy"

# Re-apply numeric conversion after generic text cleanup to keep survey stats numeric.
for c in [
    "human_verdict_agreement",
    "human_completeness_mean",
    "human_completeness_std",
    "human_completeness_agreement",
    "human_logical_mean",
    "human_logical_std",
    "human_redundancy_mean",
    "human_redundancy_std",
]:
    if c in calibration_examples.columns:
        calibration_examples[c] = calibration_examples[c].apply(parse_float)

if "human_completeness_mean" in calibration_examples.columns:
    calibration_examples["human_completeness_folded"] = calibration_examples["human_completeness_mean"].apply(fold_likert_completeness_or_logical)
if "human_logical_mean" in calibration_examples.columns:
    calibration_examples["human_logical_folded"] = calibration_examples["human_logical_mean"].apply(fold_likert_completeness_or_logical)
if "human_redundancy_mean" in calibration_examples.columns:
    calibration_examples["human_redundancy_folded"] = calibration_examples["human_redundancy_mean"].apply(fold_likert_redundancy_reverse)

# Final schema checks
rules_required = ["parent_rule_id", "parent_rule_text", "rule_id", "rule_text_atomic", "rule_figure_caption", "ambiguity_remark"]
rules_optional = ["counter_notes", "figure_id"]

calib_required = [
    "reason_gt_id", "parent_rule_id", "question_id", "scene_id", "task_family", "ground_truth_answer", "answer_to_evaluate",
    "survey_question_id", "human_verdict_distribution", "human_verdict_consensus", "human_verdict_agreement",
    "human_completeness_mean", "human_completeness_std", "human_completeness_agreement",
    "human_logical_mean", "human_logical_std", "human_redundancy_mean", "human_redundancy_std",
    "counter_note", "rule_interpretation_note", "split"
]
calib_optional = ["reasoning_pattern", "error_type", "ambiguity_level", "figure_id"]

rubric_required = ["metric_name", "definition", "score_scale", "scoring_guidelines", "anti_bias_note"]

eval_required = [
    "generated_question_id", "template_id", "question_text", "scene_id", "file_path", "figure_path", "RuleID", "ParentRuleID",
    "Classification", "Ambiguity", "ClassificationParent", "rule_figure_id", "correct_answer", "CorrectAnswer", "model_answer",
    "openrouter_request_id", "error_reason", "row_index", "bool_model_answer"
]

validate_required_columns(rules_table, rules_required, "rules_table")
optional_column_report(rules_table, rules_optional, "rules_table")
validate_required_columns(calibration_examples, calib_required, "calibration_examples")
optional_column_report(calibration_examples, calib_optional, "calibration_examples")
validate_required_columns(metric_rubrics, rubric_required, "metric_rubrics")
validate_required_columns(evaluation_items, eval_required, "evaluation_items")

print("All required schema checks passed.")
print("Calibration split counts:")
print(calibration_examples["split"].value_counts(dropna=False))


rules_table: optional missing -> ['counter_notes']
calibration_examples: optional missing -> ['reasoning_pattern', 'error_type', 'ambiguity_level']
All required schema checks passed.
Calibration split counts:
split
anchor             19
holdout             6
hard_validation     4
Name: count, dtype: int64


## 7. Embedding Preparation

Embedding text blocks are built for calibration retrieval.
These vectors support similarity ranking when selecting in-context examples for each metric call.


In [91]:
# Build helper maps and embedding texts

def combine_text(parts: List[Any]) -> str:
    out = []
    for p in parts:
        s = clean_text(p)
        if s:
            out.append(s)
    return "\n".join(out)

rule_parent_map = (
    rules_table[["rule_id", "parent_rule_id"]]
    .drop_duplicates(subset=["rule_id"])
    .set_index("rule_id")["parent_rule_id"]
    .to_dict()
)

miss_parent = evaluation_items["ParentRuleID"].eq("") & evaluation_items["RuleID"].ne("")
evaluation_items.loc[miss_parent, "ParentRuleID"] = evaluation_items.loc[miss_parent, "RuleID"].map(rule_parent_map).fillna("")

for c in ["reasoning_pattern", "error_type", "ambiguity_level"]:
    if c not in calibration_examples.columns:
        calibration_examples[c] = ""

calibration_examples["calib_embed_text"] = calibration_examples.apply(
    lambda r: combine_text([
        r.get("answer_to_evaluate", ""),
        r.get("ground_truth_answer", ""),
        r.get("task_family", ""),
        r.get("reasoning_pattern", ""),
        r.get("error_type", ""),
        r.get("counter_note", ""),
        r.get("rule_interpretation_note", ""),
    ]),
    axis=1,
)

# Split-aware calibration subsets.
# Hard constraint:
# - prompt examples in Block B come only from anchor_examples.
# - hard_validation and holdout are reserved for judge testing/validation.
anchor_examples = calibration_examples[calibration_examples["split"] == "anchor"].copy().reset_index(drop=True)
hard_validation_examples = calibration_examples[calibration_examples["split"] == "hard_validation"].copy().reset_index(drop=True)
holdout_examples = calibration_examples[calibration_examples["split"] == "holdout"].copy().reset_index(drop=True)

if anchor_examples.empty:
    raise ValueError("Calibration split policy violated: anchor_examples is empty.")

if hard_validation_examples.empty and holdout_examples.empty:
    print("WARNING: hard_validation and holdout splits are empty; split-aware test run cannot be split-grounded.")

def _as_id_set(df: pd.DataFrame, col: str) -> set:
    if col not in df.columns:
        return set()
    s = df[col].fillna("").astype(str).str.strip()
    s = s[~s.isin(["", "nan", "None", "null"])]
    return set(s.tolist())

anchor_question_ids = _as_id_set(anchor_examples, "question_id")
hard_validation_question_ids = _as_id_set(hard_validation_examples, "question_id")
holdout_question_ids = _as_id_set(holdout_examples, "question_id")
test_split_question_ids = hard_validation_question_ids | holdout_question_ids

# Parent-level priors for retrieval are learned from anchor only (no leakage).
parent_to_task_family = {}
parent_to_reasoning_pattern = {}
parent_to_error_type = {}

for pid, grp in anchor_examples.groupby("parent_rule_id", dropna=False):
    if not pid:
        continue
    tf = grp["task_family"].dropna().astype(str).str.strip()
    tf = tf[tf != ""]
    if len(tf):
        parent_to_task_family[pid] = tf.mode().iloc[0]

    if "reasoning_pattern" in grp.columns:
        rp = grp["reasoning_pattern"].dropna().astype(str).str.strip()
        rp = rp[rp != ""]
        if len(rp):
            parent_to_reasoning_pattern[pid] = rp.mode().iloc[0]

    if "error_type" in grp.columns:
        et = grp["error_type"].dropna().astype(str).str.strip()
        et = et[et != ""]
        if len(et):
            parent_to_error_type[pid] = et.mode().iloc[0]

rules_table["rule_embed_text"] = rules_table.apply(
    lambda r: combine_text([
        r.get("parent_rule_text", ""),
        r.get("rule_text_atomic", ""),
        r.get("rule_figure_caption", ""),
        r.get("ambiguity_remark", ""),
        r.get("counter_notes", ""),
    ]),
    axis=1,
)

embed_model = SentenceTransformer(EMBED_MODEL_NAME)
rule_texts = rules_table["rule_embed_text"].fillna("").astype(str).tolist()
calib_texts = anchor_examples["calib_embed_text"].fillna("").astype(str).tolist()

rule_embeddings = embed_model.encode(rule_texts, normalize_embeddings=True, show_progress_bar=True)
calib_embeddings = embed_model.encode(calib_texts, normalize_embeddings=True, show_progress_bar=True)

print("Embeddings ready:")
print("rule_embeddings:", rule_embeddings.shape)
print("calib_embeddings (anchor only):", calib_embeddings.shape)
print("Split sizes -> anchor:", len(anchor_examples), ", hard_validation:", len(hard_validation_examples), ", holdout:", len(holdout_examples))


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.52it/s]

Embeddings ready:
rule_embeddings: (69, 384)
calib_embeddings (anchor only): (19, 384)
Split sizes -> anchor: 19 , hard_validation: 4 , holdout: 6


## 8. Retrieval Functions

Retrieval uses a cascade (strict local match -> parent-rule expansion -> task-family expansion -> metric-level backfill).
Per-call diagnostics record how many examples came from each retrieval level.


In [92]:
def agreement_hint(row: pd.Series, metric_name: str) -> float:
    m = metric_name.lower()
    if "completeness" in m and "human_completeness_agreement" in row:
        return float(row.get("human_completeness_agreement") or 0.0)
    if "redundancy" in m and "human_redundancy_std" in row:
        std = row.get("human_redundancy_std")
        if std is None or (isinstance(std, float) and pd.isna(std)):
            return 0.0
        return max(0.0, 1.0 - float(std) / 2.0)
    if "commonsense" in m and "human_logical_mean" in row:
        x = row.get("human_logical_mean")
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return 0.0
        return float(x) / 5.0
    if "hallucination" in m and "human_verdict_agreement" in row:
        return float(row.get("human_verdict_agreement") or 0.0)
    return float(row.get("human_verdict_agreement") or 0.0)


def rank_pool_with_embeddings(pool: pd.DataFrame, query_vec: np.ndarray, metric_name: str, target_reasoning_pattern: str = "", target_error_type: str = "") -> pd.DataFrame:
    if pool.empty:
        return pool.copy()

    idx = pool.index.to_numpy()
    vecs = calib_embeddings[idx]
    sims = cosine_sim_1_to_many(query_vec, vecs)

    ranked = pool.copy()
    ranked["_sim"] = sims
    ranked["_agreement"] = ranked.apply(lambda r: agreement_hint(r, metric_name), axis=1)
    ranked["_boost"] = 0.0

    if target_reasoning_pattern and "reasoning_pattern" in ranked.columns:
        ranked.loc[ranked["reasoning_pattern"].str.lower() == target_reasoning_pattern.lower(), "_boost"] += 0.08

    if target_error_type and "error_type" in ranked.columns:
        ranked.loc[ranked["error_type"].str.lower() == target_error_type.lower(), "_boost"] += 0.05

    ranked["_score"] = SIM_WEIGHT * ranked["_sim"] + AGREEMENT_WEIGHT * ranked["_agreement"] + ranked["_boost"]
    ranked = ranked.sort_values(["_score", "reason_gt_id"], ascending=[False, True])
    return ranked


def _exclude_current_item_from_pool(pool: pd.DataFrame, item: pd.Series) -> pd.DataFrame:
    """Anti-leakage guard: evaluated row must not appear inside calibration prompt examples."""
    out = pool.copy()
    current_ids = set()
    for c in ["reason_gt_id", "question_id", "generated_question_id"]:
        v = normalize_id(item.get(c, ""))
        if v:
            current_ids.add(v)
    gid = normalize_id(item.get("generated_question_id", ""))
    if gid:
        current_ids.add(gid)
    if not current_ids:
        return out
    for c in ["reason_gt_id", "question_id"]:
        if c in out.columns:
            out = out[~out[c].isin(current_ids)]
    return out


def select_examples(item: pd.Series, metric_name: str) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    parent_id = item.get("ParentRuleID", "")
    scene_id = item.get("scene_id", "")

    target_task_family = parent_to_task_family.get(parent_id, "")
    target_reasoning_pattern = parent_to_reasoning_pattern.get(parent_id, "")
    target_error_type = parent_to_error_type.get(parent_id, "")

    query_text = combine_text([
        item.get("question_text", ""),
        item.get("model_answer", ""),
        target_task_family,
        target_reasoning_pattern,
        target_error_type,
    ])
    query_vec = embed_model.encode([query_text], normalize_embeddings=True)[0]

    selected_ids = set()
    selected_blocks = []

    diag = {
        "level1_n": 0,
        "level2_n": 0,
        "level3_n": 0,
        "level4_n": 0,
        "local_pool_sufficient": False,
        "used_fallback": False,
        "calibration_split_policy": "anchor_only",
    }

    # Hard split policy for Block B prompt examples:
    # - only anchor_examples are allowed as calibration context.
    # - hard_validation and holdout are never injected into judge prompt.
    calibration_pool = _exclude_current_item_from_pool(anchor_examples, item)

    def take_from_pool(pool: pd.DataFrame, level_name: str, n_take: int) -> int:
        if pool.empty or n_take <= 0:
            return 0
        ranked = rank_pool_with_embeddings(pool, query_vec, metric_name, target_reasoning_pattern, target_error_type)
        ranked = ranked[~ranked["reason_gt_id"].isin(selected_ids)].head(n_take).copy()
        if ranked.empty:
            return 0
        ranked["retrieval_level"] = level_name
        selected_blocks.append(ranked)
        selected_ids.update(ranked["reason_gt_id"].tolist())
        return len(ranked)

    level1 = calibration_pool[
        (calibration_pool["parent_rule_id"] == parent_id)
        & (calibration_pool["scene_id"] == scene_id)
    ]
    diag["level1_n"] = take_from_pool(level1, "level1_same_scene_same_parent", MAX_CALIB)

    if len(selected_ids) < MIN_LOCAL_CALIB:
        level2 = calibration_pool[
            (calibration_pool["parent_rule_id"] == parent_id)
            & (calibration_pool["scene_id"] != scene_id)
        ]
        need = MAX_CALIB - len(selected_ids)
        diag["level2_n"] = take_from_pool(level2, "level2_same_parent_other_scenes", need)

    local_count = diag["level1_n"] + diag["level2_n"]
    diag["local_pool_sufficient"] = local_count >= MIN_LOCAL_CALIB

    if len(selected_ids) < MIN_LOCAL_CALIB:
        pool = calibration_pool[calibration_pool["parent_rule_id"] != parent_id].copy()
        if target_task_family:
            pool = pool[pool["task_family"].str.lower() == target_task_family.lower()]

        if target_reasoning_pattern and "reasoning_pattern" in pool.columns:
            strict = pool[pool["reasoning_pattern"].str.lower() == target_reasoning_pattern.lower()]
            soft = pool[pool["reasoning_pattern"].str.lower() != target_reasoning_pattern.lower()]
            need = min(MAX_LEVEL3_ADDS, MAX_CALIB - len(selected_ids))
            got = take_from_pool(strict, "level3_task_family_reasoning_pattern", need)
            if got < need:
                got += take_from_pool(soft, "level3_task_family", need - got)
            diag["level3_n"] = got
        else:
            need = min(MAX_LEVEL3_ADDS, MAX_CALIB - len(selected_ids))
            diag["level3_n"] = take_from_pool(pool, "level3_task_family", need)

    if len(selected_ids) < MIN_LOCAL_CALIB:
        remaining = calibration_pool[~calibration_pool["reason_gt_id"].isin(selected_ids)].copy()

        # metric-aware backfill when metric tagging exists in calibration table
        metric_cols = [c for c in ["metric_name", "target_metric", "metric"] if c in remaining.columns]
        if metric_cols:
            mcol = metric_cols[0]
            mask = remaining[mcol].astype(str).str.lower().str.contains(metric_name.lower(), regex=False)
            metric_pool = remaining[mask]
            non_metric_pool = remaining[~mask]
            need = MAX_CALIB - len(selected_ids)
            got = take_from_pool(metric_pool, f"level4_metric_backfill:{metric_name}", need)
            if got < need:
                got += take_from_pool(non_metric_pool, f"level4_cross_rule_backfill:{metric_name}", need - got)
            diag["level4_n"] = got
        else:
            need = MAX_CALIB - len(selected_ids)
            diag["level4_n"] = take_from_pool(remaining, f"level4_cross_rule_backfill:{metric_name}", need)

    diag["used_fallback"] = (diag["level3_n"] + diag["level4_n"]) > 0

    if selected_blocks:
        out = pd.concat(selected_blocks, ignore_index=True)
    else:
        out = anchor_examples.head(0).copy()

    return out.head(MAX_CALIB), diag


def get_rule_context(item: pd.Series) -> Dict[str, Any]:
    parent_id = item.get("ParentRuleID", "")
    rule_id = item.get("RuleID", "")
    scene_id = item.get("scene_id", "")

    parent_rows = rules_table[rules_table["parent_rule_id"] == parent_id].copy()
    if rule_id:
        rule_rows = parent_rows[parent_rows["rule_id"] == rule_id].copy()
        if rule_rows.empty:
            rule_rows = parent_rows
    else:
        rule_rows = parent_rows

    parent_rule_text = parent_rows["parent_rule_text"].dropna().astype(str).head(1).tolist()
    parent_rule_text = parent_rule_text[0] if parent_rule_text else ""

    atomic_rules = rule_rows["rule_text_atomic"].dropna().astype(str).unique().tolist()
    ambiguity_notes = rule_rows["ambiguity_remark"].dropna().astype(str).unique().tolist()

    captions = rule_rows["rule_figure_caption"].dropna().astype(str) if "rule_figure_caption" in rule_rows.columns else pd.Series([], dtype=str)
    figure_caption = captions.head(1).tolist()
    figure_caption = figure_caption[0] if figure_caption else ""

    local_notes = anchor_examples[
        (anchor_examples["parent_rule_id"] == parent_id)
        & (anchor_examples["scene_id"] == scene_id)
    ]["rule_interpretation_note"].dropna().astype(str).tolist()

    parent_notes = anchor_examples[
        (anchor_examples["parent_rule_id"] == parent_id)
    ]["rule_interpretation_note"].dropna().astype(str).tolist()

    interpretation_notes = [n for n in local_notes if n.strip()]
    if not interpretation_notes:
        interpretation_notes = [n for n in parent_notes if n.strip()]
    if not interpretation_notes:
        interpretation_notes = ["No rule-specific interpretation notes available."]

    return {
        "parent_rule_text": parent_rule_text,
        "atomic_rules": atomic_rules[:8],
        "figure_caption": figure_caption,
        "interpretation_notes": interpretation_notes[:6],
        "ambiguity_notes": ambiguity_notes[:6],
    }


## 9. Prompt Construction

Prompt construction is metric-specific: one prompt, one metric, one score.
The prompt includes rule context, selected calibration examples, and JSON output schema constraints.


In [93]:
def metric_key(s: str) -> str:
    t = clean_text(s).lower()
    for ch in ["-", "–", "—", "−", "‑", "?"]:
        t = t.replace(ch, "-")
    t = re.sub(r"[^a-z0-9]+", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def build_metric_rubric(metric_name: str) -> Dict[str, str]:
    mk = metric_key(metric_name)
    r = metric_rubrics[metric_rubrics["metric_name"].apply(metric_key) == mk]
    if r.empty:
        r = metric_rubrics[metric_rubrics["metric_name"].apply(metric_key).str.contains(mk, regex=False)]
    if r.empty:
        return {"metric_name": metric_name, "definition": "", "score_scale": "", "scoring_guidelines": "", "anti_bias_note": ""}
    row = r.iloc[0]
    return {
        "metric_name": row.get("metric_name", ""),
        "definition": row.get("definition", ""),
        "score_scale": row.get("score_scale", ""),
        "scoring_guidelines": row.get("scoring_guidelines", ""),
        "anti_bias_note": row.get("anti_bias_note", ""),
    }


def label_example_quality(ex: pd.Series) -> str:
    for col in ["ground_truth_judgement", "goal_judgment", "human_verdict_consensus"]:
        if col in ex.index:
            s = str(ex.get(col, "")).strip().lower()
            if s in {"good", "bad", "borderline", "yes", "no", "not_enough"}:
                return s
    return "unknown"


def metric_schema(metric_name: str) -> Dict[str, Any]:
    base = {
        "generated_question_id": "string",
        "scene_id": "string",
        "rule_id": "string",
        "metric_name": "string",
        "score": 5,
        "max_score": 5,
        "short_rationale": "string",
        "ambiguity_flag": False,
    }
    m = metric_key(metric_name)
    if "entity grounding" in m or "visual alignment" in m:
        base["label"] = "string"
        base["evidence_used"] = "string"
    return base


def build_metric_messages(item: pd.Series, metric_name: str, rule_ctx: Dict[str, Any], examples: pd.DataFrame, retrieval_diag: Dict[str, Any]) -> List[Dict[str, str]]:
    rubric = build_metric_rubric(metric_name)

    ex_lines = []
    for _, ex in examples.iterrows():
        ex_lines.append(
            f"- level={ex.get('retrieval_level','')}; reason_gt_id={ex.get('reason_gt_id','')}; quality={label_example_quality(ex)}; "
            f"task_family={ex.get('task_family','')}; reasoning_pattern={ex.get('reasoning_pattern','')}; error_type={ex.get('error_type','')}\n"
            f"  answer_to_evaluate={clean_text(ex.get('answer_to_evaluate',''))[:350]}\n"
            f"  counter_note={clean_text(ex.get('counter_note',''))[:180]}\n"
            f"  agreement(verdict/completeness/logical/redundancy)=({ex.get('human_verdict_agreement','')}, {ex.get('human_completeness_agreement','')}, {ex.get('human_logical_mean','')}, {ex.get('human_redundancy_mean','')})"
        )

    prompt_user = f"""Evaluate ONLY this metric: {metric_name}

Block A: Rule-specific context (mandatory)
- parent_rule_text: {rule_ctx.get('parent_rule_text','')}
- atomic_rules: {rule_ctx.get('atomic_rules',[])}
- figure_caption: {rule_ctx.get('figure_caption','')}
- interpretation_notes: {rule_ctx.get('interpretation_notes',[])}
- ambiguity_notes: {rule_ctx.get('ambiguity_notes',[])}

Block B: Calibration examples (retrieval cascade)
{chr(10).join(ex_lines)}

Item to evaluate
- generated_question_id: {item.get('generated_question_id','')}
- scene_id: {item.get('scene_id','')}
- rule_id: {item.get('RuleID','')}
- question_text: {item.get('question_text','')}
- candidate_reasoning: {item.get('model_answer','')}
- reference_answer: {item.get('CorrectAnswer','') or item.get('correct_answer','')}
- retrieval_diagnostics: {retrieval_diag}

Metric rubric
{rubric}

Instructions:
- evaluate only this metric;
- do not recompute other metrics;
- use only provided context;
- do not add external facts;
- keep rationale short and evidence-based (1-2 sentences).

Return JSON ONLY with this schema:
{json.dumps(metric_schema(metric_name), ensure_ascii=False)}
"""

    prompt_system = (
        "You are a strict evaluation judge for regulatory multimodal reasoning. "
        "Return JSON only."
    )

    return [
        {"role": "system", "content": prompt_system},
        {"role": "user", "content": prompt_user},
    ]


def build_overall_messages(row: pd.Series) -> List[Dict[str, str]]:
    schema = {"overall_label": "string", "overall_summary": "string"}

    user = f"""Use the provided metric values as fixed inputs. Do not recompute them. Produce only a short overall assessment.

Input summary:
{row.to_dict()}

Rules:
- no metric recalculation;
- no new evidence;
- short summary only.

Return JSON only:
{json.dumps(schema, ensure_ascii=False)}
"""
    return [
        {"role": "system", "content": "You are an aggregation assistant. Return JSON only."},
        {"role": "user", "content": user},
    ]


## 10. Stage 1: Metric-Level Judging

For each `(generated_question_id, metric_name)`, the notebook creates one judge request and stores the normalized score/rationale.
The result table at this stage is row-based (long format), not pivoted by metric.


In [94]:
def resolve_openrouter_api_key() -> str:
    key = str(globals().get("OPENROUTER_API_KEY", "") or "").strip()
    if not key:
        key = os.getenv(OPENROUTER_API_KEY_ENV, "").strip()
    if not key:
        raise RuntimeError(
            f"OpenRouter API key is not configured. Set {OPENROUTER_API_KEY_ENV} in env or OPENROUTER_API_KEY in notebook."
        )
    globals()["OPENROUTER_API_KEY"] = key
    os.environ[OPENROUTER_API_KEY_ENV] = key
    return key


def openrouter_headers(api_key: str) -> Dict[str, str]:
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    referer = str(globals().get("OPENROUTER_HTTP_REFERER", "") or "").strip()
    x_title = str(globals().get("OPENROUTER_X_TITLE", "") or "").strip()
    if referer:
        headers["HTTP-Referer"] = referer
    if x_title:
        headers["X-Title"] = x_title
    return headers


def call_openrouter(messages: List[Dict[str, str]], request_id: str, max_tokens: int) -> Dict[str, Any]:
    api_key = resolve_openrouter_api_key()

    payload = {
        "model": JUDGE_MODEL,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": TEMPERATURE,
        "response_format": {"type": "json_object"},
        "metadata": {"request_id": request_id},
    }

    backoff = 2.0
    for attempt in range(RETRY_MAX + 1):
        r = requests.post(
            f"{OPENROUTER_BASE_URL}/chat/completions",
            headers=openrouter_headers(api_key),
            data=json.dumps(payload),
            timeout=120,
        )
        if r.status_code < 400:
            return r.json()

        retriable = r.status_code in {408, 429, 502, 503}
        if attempt >= RETRY_MAX or not retriable:
            try:
                detail = r.json()
            except Exception:
                detail = r.text
            raise RuntimeError(f"OpenRouter error {r.status_code}: {detail}")

        time.sleep(backoff)
        backoff *= 1.7

    raise RuntimeError("Unexpected retry exit")


def extract_content(resp: Dict[str, Any]) -> str:
    try:
        content = resp["choices"][0]["message"].get("content")
    except Exception:
        return ""

    if isinstance(content, str):
        return content

    if isinstance(content, list):
        parts = []
        for chunk in content:
            if isinstance(chunk, dict):
                txt = chunk.get("text") or chunk.get("content") or ""
                if txt:
                    parts.append(str(txt))
            elif isinstance(chunk, str):
                parts.append(chunk)
        return "\n".join(parts).strip()

    return str(content or "")


def parse_json_response(text: str) -> Dict[str, Any]:
    t = clean_text(text)
    if not t:
        return {}

    if t.startswith("```"):
        t = re.sub(r"^```(?:json)?\s*", "", t.strip(), flags=re.IGNORECASE)
        t = re.sub(r"\s*```$", "", t, flags=re.IGNORECASE)

    try:
        parsed = json.loads(t)
        return parsed if isinstance(parsed, dict) else {}
    except Exception:
        pass

    m = re.search(r"\{.*\}", t, flags=re.DOTALL)
    if not m:
        return {}

    try:
        parsed = json.loads(m.group(0))
        return parsed if isinstance(parsed, dict) else {}
    except Exception:
        return {}


def _coerce_score_1_5(v: Any) -> Any:
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return None
    if isinstance(v, bool):
        return 5 if v else 1
    if isinstance(v, (int, float)):
        x = float(v)
        if x < 0:
            return None
        if x <= 1:
            x = x * 5
        x = min(5.0, max(1.0, x))
        return int(round(x))

    s = clean_text(v).lower()
    if not s:
        return None
    m = re.search(r"(-?\d+(?:\.\d+)?)", s)
    if not m:
        return None
    try:
        x = float(m.group(1))
    except Exception:
        return None
    if x <= 1:
        x = x * 5
    x = min(5.0, max(1.0, x))
    return int(round(x))


def normalize_metric_payload(parsed: Dict[str, Any], metric_name: str, derived_ambiguity: bool) -> Dict[str, Any]:
    d = parsed if isinstance(parsed, dict) else {}

    score = None
    for k in ["score", "metric_score", "value", "rating", "grade"]:
        if k in d:
            score = _coerce_score_1_5(d.get(k))
            if score is not None:
                break

    max_score = d.get("max_score", 5)
    max_score = _coerce_score_1_5(max_score) or 5

    short_rationale = ""
    for k in ["short_rationale", "rationale", "reason", "explanation", "justification"]:
        val = d.get(k)
        if val is not None and str(val).strip() != "":
            short_rationale = str(val).strip()
            break

    ambiguity_flag = d.get("ambiguity_flag")
    if ambiguity_flag is None:
        ambiguity_flag = derived_ambiguity
    else:
        ambiguity_flag = bool(ambiguity_flag)

    ambiguity_level = d.get("ambiguity_level")
    if not ambiguity_level:
        ambiguity_level = "high" if ambiguity_flag else "low"

    return {
        "metric_name": d.get("metric_name") or metric_name,
        "score": score,
        "max_score": max_score,
        "short_rationale": short_rationale,
        "ambiguity_flag": ambiguity_flag,
        "ambiguity_level": ambiguity_level,
    }


def derive_ambiguity_from_context(ex_df: pd.DataFrame) -> bool:
    if ex_df.empty:
        return True
    low_agreement = False
    if "human_verdict_agreement" in ex_df.columns:
        hv = pd.to_numeric(ex_df["human_verdict_agreement"], errors="coerce")
        low_agreement = bool((hv < 0.67).fillna(False).any())

    note_cols = [c for c in ["counter_note", "rule_interpretation_note", "ambiguity_level"] if c in ex_df.columns]
    ambig_note = False
    if note_cols:
        notes = " ".join(ex_df[note_cols].fillna("").astype(str).values.ravel()).lower()
        ambig_note = any(k in notes for k in ["ambig", "multiple interpretation", "defensible"])

    return low_agreement or ambig_note


def run_stage1_metric_loop(items_df: pd.DataFrame, request_prefix: str) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    records: List[Dict[str, Any]] = []
    raw: List[Dict[str, Any]] = []

    for ridx, item in items_df.reset_index(drop=True).iterrows():
        for metric_name in JUDGE_METRICS:
            req_id = f"{request_prefix}_{ridx}_{hashlib.md5(metric_name.encode()).hexdigest()[:8]}"

            rule_ctx = get_rule_context(item)
            ex_df, ex_diag = select_examples(item, metric_name)
            derived_ambiguity = derive_ambiguity_from_context(ex_df)
            messages = build_metric_messages(item, metric_name, rule_ctx, ex_df, ex_diag)

            parsed = {}
            parsed_norm = {}
            raw_resp = {}
            err = ""

            if DRY_RUN:
                parsed = {
                    "generated_question_id": item.get("generated_question_id", ""),
                    "scene_id": item.get("scene_id", ""),
                    "rule_id": item.get("RuleID", ""),
                    "metric_name": metric_name,
                    "score": None,
                    "max_score": 5,
                    "short_rationale": "DRY_RUN placeholder",
                    "ambiguity_flag": derived_ambiguity,
                }
                parsed_norm = normalize_metric_payload(parsed, metric_name, derived_ambiguity)
            else:
                try:
                    raw_resp = call_openrouter(messages, req_id, MAX_TOKENS_STAGE1)
                    parsed = parse_json_response(extract_content(raw_resp))
                    parsed_norm = normalize_metric_payload(parsed, metric_name, derived_ambiguity)
                except Exception as e:
                    err = str(e)
                    parsed_norm = normalize_metric_payload({}, metric_name, derived_ambiguity)

            rec = {
                "generated_question_id": item.get("generated_question_id", ""),
                "template_id": item.get("template_id", ""),
                "scene_id": item.get("scene_id", ""),
                "RuleID": item.get("RuleID", ""),
                "ParentRuleID": item.get("ParentRuleID", ""),
                "candidate_model": item.get("candidate_model", ""),
                "source_eval_file": item.get("source_eval_file", ""),
                "judge_model": JUDGE_MODEL,
                "metric_name": parsed_norm.get("metric_name", metric_name),
                "score": parsed_norm.get("score"),
                "max_score": parsed_norm.get("max_score", 5),
                "short_rationale": parsed_norm.get("short_rationale", ""),
                "ambiguity_flag": bool(parsed_norm.get("ambiguity_flag", derived_ambiguity)),
                "ambiguity_level": parsed_norm.get("ambiguity_level", "high" if derived_ambiguity else "low"),
                "retrieval_level1_n": ex_diag.get("level1_n", 0),
                "retrieval_level2_n": ex_diag.get("level2_n", 0),
                "retrieval_level3_n": ex_diag.get("level3_n", 0),
                "retrieval_level4_n": ex_diag.get("level4_n", 0),
                "local_pool_sufficient": ex_diag.get("local_pool_sufficient", False),
                "used_fallback": ex_diag.get("used_fallback", False),
                "retrieved_examples_n": int(len(ex_df)),
                "openrouter_request_id": req_id,
                "error_reason": err,
            }
            records.append(rec)

            raw.append({
                "request_id": req_id,
                "metric_name": metric_name,
                "generated_question_id": item.get("generated_question_id", ""),
                "candidate_model": item.get("candidate_model", ""),
                "source_eval_file": item.get("source_eval_file", ""),
                "retrieval_diag": ex_diag,
                "prompt_messages": messages,
                "response": raw_resp,
                "parsed": parsed,
                "parsed_normalized": parsed_norm,
                "error": err,
            })

    return records, raw


## 10.1. Preview Sample Before Final Run

This is a strict preview run on 5 eligible items with real API calls.
It is used to verify score parsing, request logging, and metric behavior before full execution.
Full-dataset processing remains blocked until `FULL_RUN = True` is set explicitly.


In [95]:
TEST5_N_ITEMS = 5
FULL_RUN = False  # hard safety gate for downstream full-dataset cells

TEST5_STAGE1_METRIC_CSV = OUT_DIR / "3_2_to_3_5_stage1_metric_scores_preview.csv"
TEST5_STAGE2_OVERALL_CSV = OUT_DIR / "3_2_to_3_5_test_stage2_overall.csv"
TEST5_RAW_JSONL = OUT_DIR / "3_2_to_3_5_test_raw_openrouter.jsonl"
TEST5_COMPARE_CSV = OUT_DIR / "3_2_to_3_5_test_stage1_vs_reference.csv"


def _non_empty_mask(df: pd.DataFrame, col: str) -> pd.Series:
    if col not in df.columns:
        return pd.Series(True, index=df.index)
    return df[col].fillna("").astype(str).str.strip().ne("")


# TEST MODE uses calibration CSV splits directly as evaluation targets.
# Hard split policy:
# - anchor rows are only prompt-side calibration examples.
# - hard_validation/holdout rows are judge evaluation targets.
if hard_validation_examples.empty and holdout_examples.empty:
    raise ValueError("No hard_validation/holdout rows available for TEST MODE.")

test_targets_pool = pd.concat([
    hard_validation_examples.assign(calibration_split_target="hard_validation"),
    holdout_examples.assign(calibration_split_target="holdout"),
], ignore_index=True)

# Defensive guard: test target pool must not contain anchor rows.
if "split" in test_targets_pool.columns:
    bad_split_rows = test_targets_pool[~test_targets_pool["split"].isin(["hard_validation", "holdout"])]
    if not bad_split_rows.empty:
        raise ValueError("Split policy violation: non-test split rows leaked into test target pool.")

required_test_cols = ["reason_gt_id", "question_id", "question_text", "answer_to_evaluate", "parent_rule_id", "scene_id"]
missing_test_cols = [c for c in required_test_cols if c not in test_targets_pool.columns]
if missing_test_cols:
    raise ValueError(f"Calibration test pool is missing required columns: {missing_test_cols}")

eligible_test_mask = (
    _non_empty_mask(test_targets_pool, "reason_gt_id")
    & _non_empty_mask(test_targets_pool, "question_text")
    & _non_empty_mask(test_targets_pool, "answer_to_evaluate")
    & _non_empty_mask(test_targets_pool, "parent_rule_id")
    & _non_empty_mask(test_targets_pool, "scene_id")
)

test_targets_pool = test_targets_pool[eligible_test_mask].copy()
if test_targets_pool.empty:
    raise ValueError("No eligible hard_validation/holdout rows after basic filtering.")

# Build judge-evaluation items directly from calibration rows (NOT from production evaluation_items).
test_targets_pool = test_targets_pool.sort_values(["calibration_split_target", "question_id", "reason_gt_id"]).reset_index(drop=True)
test_targets = test_targets_pool.head(TEST5_N_ITEMS).copy()
if len(test_targets) != TEST5_N_ITEMS:
    raise ValueError(f"Need exactly {TEST5_N_ITEMS} test targets from calibration splits, found {len(test_targets)}")

test5_items = pd.DataFrame({
    "generated_question_id": test_targets["reason_gt_id"].astype(str),
    "template_id": "",
    "scene_id": test_targets["scene_id"].astype(str),
    "RuleID": "",
    "ParentRuleID": test_targets["parent_rule_id"].astype(str),
    "question_text": test_targets["question_text"].astype(str),
    "model_answer": test_targets["answer_to_evaluate"].astype(str),
    "CorrectAnswer": test_targets.get("ground_truth_answer", "").astype(str) if "ground_truth_answer" in test_targets.columns else "",
    "correct_answer": test_targets.get("ground_truth_answer", "").astype(str) if "ground_truth_answer" in test_targets.columns else "",
    "candidate_model": "calibration_test_target",
    "source_eval_file": CALIBRATION_CSV.name,
    "calibration_split_target": test_targets["calibration_split_target"].astype(str),
    "reason_gt_id": test_targets["reason_gt_id"].astype(str),
    "question_id": test_targets["question_id"].astype(str),
    "goal_judgment": test_targets.get("goal_judgment", "").astype(str) if "goal_judgment" in test_targets.columns else "",
    "ground_truth_judgement": test_targets.get("ground_truth_judgement", "").astype(str) if "ground_truth_judgement" in test_targets.columns else "",
})

for c in ["human_completeness_mean", "human_logical_mean", "human_redundancy_mean", "human_verdict_consensus"]:
    if c in test_targets.columns:
        test5_items[c] = test_targets[c].values


def extract_expected_scores_from_reference(text: Any) -> Dict[str, float]:
    t = clean_text(text)
    if not t:
        return {}

    out: Dict[str, float] = {}

    def _pick(metric_name: str, patterns: List[str], binary_01: bool = False) -> None:
        for p in patterns:
            m = re.search(p, t, flags=re.IGNORECASE)
            if not m:
                continue
            try:
                val = float(m.group(1))
            except Exception:
                continue
            if binary_01:
                val = 5.0 if val >= 1 else 1.0
            out[metric_name] = max(1.0, min(5.0, val))
            return

    _pick("Hallucination", [r"hallucination\s*:\s*([0-9]+(?:\.[0-9]+)?)\s*/\s*5"]) 
    _pick("Redundancy", [r"redundancy\s*:\s*([0-9]+(?:\.[0-9]+)?)\s*/\s*5"]) 
    _pick("Commonsense", [r"commonsense\s*:\s*([0-9]+(?:\.[0-9]+)?)\s*/\s*5", r"logical coherence\s*:\s*([0-9]+(?:\.[0-9]+)?)\s*/\s*5"]) 
    _pick("Completeness", [r"completeness\s*:\s*([0-9]+(?:\.[0-9]+)?)\s*/\s*5"]) 
    _pick("CRCS - Entity Grounding", [r"entity grounding\s*:\s*([0-9]+(?:\.[0-9]+)?)\s*/\s*5"]) 
    _pick("CRCS - Visual Alignment", [r"visual alignment\s*:\s*([0-9]+(?:\.[0-9]+)?)\s*/\s*5"]) 
    _pick("CRCS - Reference Mention Accuracy", [r"reference mention accuracy\s*=\s*([0-9]+(?:\.[0-9]+)?)\s*/\s*1", r"reference mention accuracy\s*:\s*([0-9]+(?:\.[0-9]+)?)\s*/\s*1"], binary_01=True)

    return out


expected_rows = []
for _, row in test5_items.iterrows():
    gid = row.get("generated_question_id", "")
    model = row.get("candidate_model", "")

    ref_scores = extract_expected_scores_from_reference(row.get("ground_truth_judgement", ""))
    for metric_name, expected_score in ref_scores.items():
        expected_rows.append({
            "generated_question_id": gid,
            "candidate_model": model,
            "metric_name": metric_name,
            "expected_score": float(expected_score),
            "expected_source": "ground_truth_judgement",
        })

    cm = parse_float(row.get("human_completeness_mean"))
    if cm is not None:
        expected_rows.append({
            "generated_question_id": gid,
            "candidate_model": model,
            "metric_name": "Completeness",
            "expected_score": float(cm),
            "expected_source": "human_completeness_mean",
        })

    lm = parse_float(row.get("human_logical_mean"))
    if lm is not None:
        expected_rows.append({
            "generated_question_id": gid,
            "candidate_model": model,
            "metric_name": "Commonsense",
            "expected_score": float(lm),
            "expected_source": "human_logical_mean",
        })

    rm = parse_float(row.get("human_redundancy_mean"))
    if rm is not None:
        expected_rows.append({
            "generated_question_id": gid,
            "candidate_model": model,
            "metric_name": "Redundancy",
            "expected_score": float(rm),
            "expected_source": "human_redundancy_mean",
        })

expected_metric_ref_df = pd.DataFrame(expected_rows)
if not expected_metric_ref_df.empty:
    expected_metric_ref_df = expected_metric_ref_df.sort_values(
        ["generated_question_id", "candidate_model", "metric_name", "expected_source"]
    ).drop_duplicates(subset=["generated_question_id", "candidate_model", "metric_name"], keep="first")

input_show_cols = [
    "generated_question_id", "reason_gt_id", "question_id", "calibration_split_target", "scene_id", "ParentRuleID",
    "question_text", "model_answer", "correct_answer", "goal_judgment", "candidate_model", "source_eval_file",
]
input_show_cols = [c for c in input_show_cols if c in test5_items.columns]

print(f"Real test run: {TEST5_N_ITEMS} items, one judge call per metric.")
print(f"Judge model: {JUDGE_MODEL}")
print("Split policy: prompt calibration examples=anchor only; test targets=hard_validation/holdout only")
print("Test split distribution:")
print(test5_items["calibration_split_target"].value_counts(dropna=False))
print(f"Reference rows for metric comparison: {len(expected_metric_ref_df)}")
api_key_test = resolve_openrouter_api_key()
print(f"OpenRouter API key configured: ***{api_key_test[-6:]}")
display(test5_items[input_show_cols])

_prev_dry_run = DRY_RUN
DRY_RUN = False  # real OpenRouter calls for strict 5-item test run

stage1_raw_rows = []
stage2_raw_rows = []

try:
    stage1_records, stage1_raw_rows = run_stage1_metric_loop(test5_items, "test5_stage1")
    test5_stage1_metric_df = pd.DataFrame(stage1_records)

    st1 = test5_stage1_metric_df.copy()
    if st1.empty:
        raise ValueError("Stage 1 returned zero metric rows in test run.")

    for c in ["score", "max_score"]:
        if c in st1.columns:
            st1[c] = pd.to_numeric(st1[c], errors="coerce")

    pivot_index = [c for c in ["generated_question_id", "candidate_model", "judge_model", "scene_id", "RuleID", "ParentRuleID"] if c in st1.columns]
    metric_pivot = st1.pivot_table(index=pivot_index, columns="metric_name", values="score", aggfunc="first").reset_index()

    src_cols = [c for c in ["generated_question_id", "candidate_model", "model_answer", "CorrectAnswer", "correct_answer"] if c in test5_items.columns]
    src = test5_items[src_cols].drop_duplicates(subset=[c for c in ["generated_question_id", "candidate_model"] if c in src_cols])

    merge_keys = [c for c in ["generated_question_id", "candidate_model"] if c in metric_pivot.columns and c in src.columns]
    if merge_keys:
        test5_stage1_merged_df = metric_pivot.merge(src, on=merge_keys, how="left")
    else:
        test5_stage1_merged_df = metric_pivot.copy()

    def _yn(v: Any) -> str:
        s = clean_text(v).lower()
        if s in {"yes", "true", "1", "@yes"}:
            return "yes"
        if s in {"no", "false", "0", "@no"}:
            return "no"
        return ""

    def _parse_model_yes_no(text: Any) -> str:
        t = clean_text(text).lower()
        if "@yes" in t:
            return "yes"
        if "@no" in t:
            return "no"
        if re.search(r"\byes\b", t):
            return "yes"
        if re.search(r"\bno\b", t):
            return "no"
        return ""

    def _bool_to_5(x: int) -> int:
        return 5 if int(x) == 1 else 1

    def _to_unit_0_1(x: Any) -> float:
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return np.nan
        try:
            x = float(x)
        except Exception:
            return np.nan
        return max(0.0, min(1.0, (x - 1.0) / 4.0))

    if "model_answer" in test5_stage1_merged_df.columns:
        pred_yn = test5_stage1_merged_df["model_answer"].apply(_parse_model_yes_no)
    else:
        pred_yn = pd.Series("", index=test5_stage1_merged_df.index)

    if "CorrectAnswer" in test5_stage1_merged_df.columns:
        truth_series = test5_stage1_merged_df["CorrectAnswer"].copy()
    else:
        truth_series = pd.Series("", index=test5_stage1_merged_df.index)

    if "correct_answer" in test5_stage1_merged_df.columns:
        fallback_truth = test5_stage1_merged_df["correct_answer"]
        truth_series = truth_series.where(truth_series.astype(str).str.strip() != "", fallback_truth)

    true_yn = truth_series.apply(_yn)
    final_verdict_acc = ((pred_yn == true_yn) & (true_yn != "")).astype(int)
    test5_stage1_merged_df["CRCS - Final Verdict Accuracy"] = final_verdict_acc.apply(_bool_to_5)

    if "CRCS - Reference Mention Accuracy" not in test5_stage1_merged_df.columns:
        test5_stage1_merged_df["CRCS - Reference Mention Accuracy"] = np.nan

    crcs_components = [
        "CRCS - Final Verdict Accuracy",
        "CRCS - Reference Mention Accuracy",
        "CRCS - Entity Grounding",
        "CRCS - Visual Alignment",
    ]
    for c in crcs_components:
        if c not in test5_stage1_merged_df.columns:
            test5_stage1_merged_df[c] = np.nan

    unit_components = np.column_stack([test5_stage1_merged_df[c].apply(_to_unit_0_1).to_numpy() for c in crcs_components])
    test5_stage1_merged_df["CRCS_deterministic_unit"] = np.nanmean(unit_components, axis=1)
    test5_stage1_merged_df["CRCS_deterministic_score_1_5"] = (test5_stage1_merged_df["CRCS_deterministic_unit"] * 4 + 1).round(3)

    for c in ["Completeness", "Commonsense", "Redundancy"]:
        if c not in test5_stage1_merged_df.columns:
            test5_stage1_merged_df[c] = np.nan

    comp = test5_stage1_merged_df["Completeness"].apply(_to_unit_0_1)
    comm = test5_stage1_merged_df["Commonsense"].apply(_to_unit_0_1)
    red_inv = 1 - test5_stage1_merged_df["Redundancy"].apply(_to_unit_0_1)
    test5_stage1_merged_df["RSF_deterministic_unit"] = np.nanmean(np.column_stack([comp, comm, red_inv]), axis=1)
    test5_stage1_merged_df["RSF_deterministic_score_1_5"] = (test5_stage1_merged_df["RSF_deterministic_unit"] * 4 + 1).round(3)

    amb_key_cols = [c for c in ["generated_question_id", "candidate_model"] if c in st1.columns]
    if amb_key_cols:
        st1_amb = st1.groupby(amb_key_cols, as_index=False)["ambiguity_flag"].max()
        test5_stage1_merged_df = test5_stage1_merged_df.merge(st1_amb, on=amb_key_cols, how="left")

    overall_rows = []
    for _, row in test5_stage1_merged_df.iterrows():
        req_id = f"test5_stage2_{row.get('generated_question_id','')}_{hashlib.md5(str(row.get('candidate_model','')).encode()).hexdigest()[:8]}"

        payload = {
            "generated_question_id": row.get("generated_question_id", ""),
            "candidate_model": row.get("candidate_model", ""),
            "judge_model": row.get("judge_model", ""),
            "metric_values": {
                "Hallucination": row.get("Hallucination", None),
                "Redundancy": row.get("Redundancy", None),
                "Commonsense": row.get("Commonsense", None),
                "Completeness": row.get("Completeness", None),
                "CRCS - Entity Grounding": row.get("CRCS - Entity Grounding", None),
                "CRCS - Visual Alignment": row.get("CRCS - Visual Alignment", None),
                "CRCS - Final Verdict Accuracy": row.get("CRCS - Final Verdict Accuracy", None),
                "CRCS - Reference Mention Accuracy": row.get("CRCS - Reference Mention Accuracy", None),
                "CRCS_deterministic_score_1_5": row.get("CRCS_deterministic_score_1_5", None),
                "RSF_deterministic_score_1_5": row.get("RSF_deterministic_score_1_5", None),
            },
            "ambiguity_flag": bool(row.get("ambiguity_flag", False)),
        }

        parsed = {}
        err = ""
        raw_resp = {}

        try:
            msgs = build_overall_messages(pd.Series(payload))
            raw_resp = call_openrouter(msgs, req_id, MAX_TOKENS_STAGE2)
            parsed = parse_json_response(extract_content(raw_resp))
        except Exception as e:
            err = str(e)

        overall_rows.append({
            "generated_question_id": row.get("generated_question_id", ""),
            "candidate_model": row.get("candidate_model", ""),
            "judge_model": row.get("judge_model", ""),
            "scene_id": row.get("scene_id", ""),
            "rule_id": row.get("RuleID", row.get("rule_id", "")),
            "overall_label": parsed.get("overall_label", ""),
            "overall_summary": parsed.get("overall_summary", ""),
            "openrouter_request_id": req_id,
            "error_reason": err,
        })

        stage2_raw_rows.append({
            "request_id": req_id,
            "stage": "stage2_overall",
            "generated_question_id": row.get("generated_question_id", ""),
            "candidate_model": row.get("candidate_model", ""),
            "payload": payload,
            "response": raw_resp,
            "parsed": parsed,
            "error": err,
        })

    test5_stage2_overall_df = pd.DataFrame(overall_rows)
finally:
    DRY_RUN = _prev_dry_run

OUT_DIR.mkdir(parents=True, exist_ok=True)
test5_stage1_metric_df.to_csv(TEST5_STAGE1_METRIC_CSV, index=False)
test5_stage2_overall_df.to_csv(TEST5_STAGE2_OVERALL_CSV, index=False)

with open(TEST5_RAW_JSONL, "w", encoding="utf-8") as f:
    for r in stage1_raw_rows:
        out = dict(r)
        out["stage"] = "stage1_metric"
        f.write(json.dumps(out, ensure_ascii=False) + "\n")
    for r in stage2_raw_rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

stage1_display_df = test5_stage1_metric_df.copy()
if "RuleID" in stage1_display_df.columns and "rule_id" not in stage1_display_df.columns:
    stage1_display_df["rule_id"] = stage1_display_df["RuleID"]

stage1_compare_df = stage1_display_df.copy()
if not expected_metric_ref_df.empty:
    stage1_compare_df = stage1_compare_df.merge(
        expected_metric_ref_df,
        on=["generated_question_id", "candidate_model", "metric_name"],
        how="left",
    )
else:
    stage1_compare_df["expected_score"] = np.nan
    stage1_compare_df["expected_source"] = ""

stage1_compare_df["score_num"] = pd.to_numeric(stage1_compare_df.get("score"), errors="coerce")
stage1_compare_df["expected_score_num"] = pd.to_numeric(stage1_compare_df.get("expected_score"), errors="coerce")
stage1_compare_df["abs_error"] = (stage1_compare_df["score_num"] - stage1_compare_df["expected_score_num"]).abs()
stage1_compare_df["exact_match"] = (stage1_compare_df["score_num"].round(0) == stage1_compare_df["expected_score_num"].round(0))
stage1_compare_df.to_csv(TEST5_COMPARE_CSV, index=False)

required_display_cols = [
    "generated_question_id", "metric_name", "score", "expected_score", "expected_source", "abs_error",
    "short_rationale", "ambiguity_flag", "candidate_model", "rule_id", "scene_id", "error_reason",
]
available_display_cols = [c for c in required_display_cols if c in stage1_compare_df.columns]

print(f"Saved stage1 metric preview to: {TEST5_STAGE1_METRIC_CSV}")
print(f"Saved stage1 vs reference comparison to: {TEST5_COMPARE_CSV}")
print(f"Saved stage2 overall preview to: {TEST5_STAGE2_OVERALL_CSV}")
print(f"Saved raw OpenRouter logs to: {TEST5_RAW_JSONL}")
print("Displayed rows are real judge outputs for the strict 5-item split-aware test run.")


stage1_total_rows = len(test5_stage1_metric_df)
stage1_scored_rows = int(pd.to_numeric(test5_stage1_metric_df.get("score"), errors="coerce").notna().sum()) if stage1_total_rows else 0
print(f"Stage1 preview rows: {stage1_total_rows}; scored rows: {stage1_scored_rows}")
if stage1_total_rows and stage1_scored_rows == 0:
    print("WARNING: all preview metric scores are empty. Check error_reason and raw OpenRouter JSONL.")

display(stage1_compare_df[available_display_cols])
display(test5_stage2_overall_df)


Real test run: 5 items, one judge call per metric.
Judge model: anthropic/claude-sonnet-4.5
Split policy: prompt calibration examples=anchor only; test targets=hard_validation/holdout only
Test split distribution:
calibration_split_target
hard_validation    4
holdout            1
Name: count, dtype: int64
Reference rows for metric comparison: 24
OpenRouter API key configured: ***8934ce


,generated_question_id,reason_gt_id,question_id,calibration_split_target,scene_id,ParentRuleID,question_text,model_answer,correct_answer,goal_judgment,candidate_model,source_eval_file
0,7,7,16,hard_validation,1,2,Is the design displayed in the picture complia...,Step 1: Identify the reference standard and es...,Step 1: Identify the reference standard (Figur...,"unmet,\ncorrect_flawed_reasoning",calibration_test_target,03_04_reasonLLMjudgeCalibr.csv
1,2,2,246,hard_validation,26,2,Is the design displayed in the picture complia...,Step 1: Identify the reference standard (Figur...,Step 1: Identify the reference standard (Figur...,"unmet,\ncorrect_flawed_reasoning",calibration_test_target,03_04_reasonLLMjudgeCalibr.csv
2,10,10,36,hard_validation,9,1,Is the design displayed in the picture complia...,Step 1: Identify the rule requirements and key...,Step 1: Identify the rule requirements and key...,"unmet,\ncorrect_flawed_reasoning",calibration_test_target,03_04_reasonLLMjudgeCalibr.csv
3,15,15,50,hard_validation,11,1,Is the design displayed in the picture complia...,Step 1: Identify door types and locations in t...,Step 1: Identify the rule requirements and key...,"unmet,\ncorrect_flawed_reasoning",calibration_test_target,03_04_reasonLLMjudgeCalibr.csv
4,17,17,175,holdout,19,6,Is the design displayed in the picture complia...,Step 1: Identify the rule requirements\nStep 2...,,met,calibration_test_target,03_04_reasonLLMjudgeCalibr.csv


Saved stage1 metric preview to: ..\results\3_2_to_3_5_stage1_metric_scores_preview.csv
Saved stage1 vs reference comparison to: ..\results\3_2_to_3_5_test_stage1_vs_reference.csv
Saved stage2 overall preview to: ..\results\3_2_to_3_5_test_stage2_overall.csv
Saved raw OpenRouter logs to: ..\results\3_2_to_3_5_test_raw_openrouter.jsonl
Displayed rows are real judge outputs for the strict 5-item split-aware test run.
Stage1 preview rows: 35; scored rows: 35


,generated_question_id,metric_name,score,expected_score,expected_source,abs_error,short_rationale,ambiguity_flag,candidate_model,rule_id,scene_id,error_reason
0,7,Hallucination,3,4.0,ground_truth_judgement,1.0,Several invented details undermine grounding: ...,False,calibration_test_target,,1,
1,7,Redundancy,3,5.0,ground_truth_judgement,2.0,Step 1 unnecessarily separates primary and sec...,False,calibration_test_target,,1,
2,7,Commonsense,5,3.0,ground_truth_judgement,2.0,The reasoning demonstrates adequate commonsens...,False,calibration_test_target,,1,
3,7,Completeness,5,5.0,ground_truth_judgement,0.0,All critical reasoning steps are present: refe...,False,calibration_test_target,,1,
4,7,CRCS – Reference Mention Accuracy,5,NaN,NaN,NaN,The candidate reasoning explicitly references ...,False,calibration_test_target,,1,
5,7,CRCS – Entity Grounding,5,NaN,NaN,NaN,"All key entities (toilet fixture, hand-rinse b...",False,calibration_test_target,,1,
6,7,CRCS – Visual Alignment,4,NaN,NaN,NaN,The candidate correctly identifies key visual ...,False,calibration_test_target,,1,
7,2,Hallucination,4,5.0,ground_truth_judgement,1.0,The candidate reasoning is well-grounded in th...,False,calibration_test_target,,26,
8,2,Redundancy,4,4.0,ground_truth_judgement,0.0,The reasoning is well-structured with minimal ...,False,calibration_test_target,,26,
9,2,Commonsense,5,3.0,ground_truth_judgement,2.0,The candidate reasoning applies necessary comm...,False,calibration_test_target,,26,


,generated_question_id,candidate_model,judge_model,scene_id,rule_id,overall_label,overall_summary,openrouter_request_id,error_reason
0,10,calibration_test_target,anthropic/claude-sonnet-4.5,9,,Poor,The response shows significant issues with hal...,test5_stage2_10_c9148b0c,
1,15,calibration_test_target,anthropic/claude-sonnet-4.5,11,,Poor,The response shows significant issues with a C...,test5_stage2_15_c9148b0c,
2,17,calibration_test_target,anthropic/claude-sonnet-4.5,19,,Poor,The response demonstrates critical failures wi...,test5_stage2_17_c9148b0c,
3,2,calibration_test_target,anthropic/claude-sonnet-4.5,26,,Poor,The response shows critical failures in accura...,test5_stage2_2_c9148b0c,
4,7,calibration_test_target,anthropic/claude-sonnet-4.5,1,,Poor,The response shows critical failures in accura...,test5_stage2_7_c9148b0c,


In [96]:
if not globals().get("FULL_RUN", False):
    raise SystemExit(
        "Full dataset run is blocked after 10.1 test run. "
        "Set FULL_RUN=True manually and rerun this cell to execute full Stage 1."
    )

stage1_records, raw_rows = run_stage1_metric_loop(evaluation_items, "stage1")
stage1_metric_df = pd.DataFrame(stage1_records)
print("Stage 1 rows:", len(stage1_metric_df))
stage1_metric_df.head()


SystemExit: Full dataset run is blocked after 10.1 test run. Set FULL_RUN=True manually and rerun this cell to execute full Stage 1.

C:\Users\ritaMZ\miniconda3\envs\aqa-data\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 11. Stage 1b: Deterministic Composite Metric Calculation

Composite metrics are computed locally from Stage-1 outputs.

1. Final Verdict Accuracy (deterministic):
- parse model yes/no from `model_answer`;
- parse ground truth yes/no from `CorrectAnswer` (fallback `correct_answer`);
- exact match -> 5, mismatch -> 1.

2. CRCS deterministic score:
- components: `CRCS - Final Verdict Accuracy`, `CRCS - Reference Mention Accuracy`, `CRCS - Entity Grounding`, `CRCS - Visual Alignment`;
- each component is converted from 1-5 to unit scale: `(x - 1) / 4`;
- average available unit components (`nanmean`), then map back to 1-5: `unit * 4 + 1`.

3. RSF deterministic score:
- proxy unit score = mean(`Completeness_norm`, `Commonsense_norm`, `1 - Redundancy_norm`);
- then map back to 1-5 with `unit * 4 + 1`.


In [ ]:
def yn(v: Any) -> str:
    s = clean_text(v).lower()
    if s in {"yes", "true", "1", "@yes"}:
        return "yes"
    if s in {"no", "false", "0", "@no"}:
        return "no"
    return ""


def parse_model_yes_no(text: Any) -> str:
    t = clean_text(text).lower()
    if "@yes" in t:
        return "yes"
    if "@no" in t:
        return "no"
    # fallback whole-word search
    if re.search(r"\byes\b", t):
        return "yes"
    if re.search(r"\bno\b", t):
        return "no"
    return ""


def bool_to_5(x: int) -> int:
    return 5 if int(x) == 1 else 1

# Prepare metric pivot
st1 = stage1_metric_df.copy()
for c in ["score", "max_score"]:
    st1[c] = pd.to_numeric(st1[c], errors="coerce")

metric_pivot = st1.pivot_table(
    index=["generated_question_id", "candidate_model", "judge_model", "scene_id", "RuleID", "ParentRuleID"],
    columns="metric_name",
    values="score",
    aggfunc="first"
).reset_index()

# Merge source eval fields needed for deterministic components
src_cols = ["generated_question_id", "candidate_model", "model_answer", "CorrectAnswer", "correct_answer"]
src = evaluation_items[src_cols].drop_duplicates(subset=["generated_question_id", "candidate_model"])
stage1_merged_df = metric_pivot.merge(src, on=["generated_question_id", "candidate_model"], how="left")

# Deterministic component: CRCS - Final Verdict Accuracy
pred_yn = stage1_merged_df["model_answer"].apply(parse_model_yes_no)
true_yn = stage1_merged_df["CorrectAnswer"].where(stage1_merged_df["CorrectAnswer"].astype(str).str.strip() != "", stage1_merged_df["correct_answer"]).apply(yn)
final_verdict_acc = (pred_yn == true_yn).astype(int)
stage1_merged_df["CRCS - Final Verdict Accuracy"] = final_verdict_acc.apply(bool_to_5)

# CRCS - Reference Mention Accuracy is judge-scored in Stage 1.
# We do not compute it heuristically in Stage 1b.
if "CRCS - Reference Mention Accuracy" not in stage1_merged_df.columns:
    stage1_merged_df["CRCS - Reference Mention Accuracy"] = np.nan

# Composite CRCS (mean of normalized components)
crcs_components = [
    "CRCS - Final Verdict Accuracy",
    "CRCS - Reference Mention Accuracy",
    "CRCS - Entity Grounding",
    "CRCS - Visual Alignment",
]
for c in crcs_components:
    if c not in stage1_merged_df.columns:
        stage1_merged_df[c] = np.nan

def to_unit_0_1(x: Any) -> float:
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return np.nan
    x = float(x)
    return max(0.0, min(1.0, (x - 1.0) / 4.0))

unit_components = np.column_stack([stage1_merged_df[c].apply(to_unit_0_1).to_numpy() for c in crcs_components])
stage1_merged_df["CRCS_deterministic_unit"] = np.nanmean(unit_components, axis=1)
stage1_merged_df["CRCS_deterministic_score_1_5"] = (stage1_merged_df["CRCS_deterministic_unit"] * 4 + 1).round(3)

# Deterministic RSF from available atomic metrics (no direct judge call)
# RSF proxy = mean(Completeness_norm, Commonsense_norm, (1 - Redundancy_norm))
for c in ["Completeness", "Commonsense", "Redundancy"]:
    if c not in stage1_merged_df.columns:
        stage1_merged_df[c] = np.nan

comp = stage1_merged_df["Completeness"].apply(to_unit_0_1)
comm = stage1_merged_df["Commonsense"].apply(to_unit_0_1)
red_inv = 1 - stage1_merged_df["Redundancy"].apply(to_unit_0_1)
stage1_merged_df["RSF_deterministic_unit"] = np.nanmean(np.column_stack([comp, comm, red_inv]), axis=1)
stage1_merged_df["RSF_deterministic_score_1_5"] = (stage1_merged_df["RSF_deterministic_unit"] * 4 + 1).round(3)

# Preserve ambiguity support
st1_amb = st1.groupby(["generated_question_id", "candidate_model"], as_index=False)["ambiguity_flag"].max()
stage1_merged_df = stage1_merged_df.merge(st1_amb, on=["generated_question_id", "candidate_model"], how="left")

stage1b_composite_df = stage1_merged_df[[
    "generated_question_id", "candidate_model", "judge_model",
    "CRCS - Final Verdict Accuracy", "CRCS - Reference Mention Accuracy",
    "CRCS - Entity Grounding", "CRCS - Visual Alignment",
    "CRCS_deterministic_score_1_5",
    "RSF_deterministic_score_1_5",
    "ambiguity_flag",
]].copy()

print("Stage 1b rows:", len(stage1b_composite_df))
stage1b_composite_df.head()


## 12. Stage 2: Overall Assessment

Stage 2 reads fixed metric values from Stage 1/1b and produces an overall label/summary.
No metric is recalculated in this stage; it is an interpretation layer over existing metric outputs.


In [ ]:
overall_rows = []

for _, row in stage1_merged_df.iterrows():
    req_id = f"stage2_{row.get('generated_question_id','')}_{hashlib.md5(str(row.get('candidate_model','')).encode()).hexdigest()[:8]}"

    payload = {
        "generated_question_id": row.get("generated_question_id", ""),
        "candidate_model": row.get("candidate_model", ""),
        "judge_model": row.get("judge_model", ""),
        "metric_values": {
            "Hallucination": row.get("Hallucination", None),
            "Redundancy": row.get("Redundancy", None),
            "Commonsense": row.get("Commonsense", None),
            "Completeness": row.get("Completeness", None),
            "CRCS - Entity Grounding": row.get("CRCS - Entity Grounding", None),
            "CRCS - Visual Alignment": row.get("CRCS - Visual Alignment", None),
            "CRCS - Final Verdict Accuracy": row.get("CRCS - Final Verdict Accuracy", None),
            "CRCS - Reference Mention Accuracy": row.get("CRCS - Reference Mention Accuracy", None),
            "CRCS_deterministic_score_1_5": row.get("CRCS_deterministic_score_1_5", None),
            "RSF_deterministic_score_1_5": row.get("RSF_deterministic_score_1_5", None),
        },
        "ambiguity_flag": bool(row.get("ambiguity_flag", False)),
    }

    parsed = {}
    err = ""
    raw_resp = {}

    if DRY_RUN:
        parsed = {
            "overall_label": "DRY_RUN",
            "overall_summary": "DRY_RUN placeholder summary from fixed metric inputs.",
        }
    else:
        try:
            msgs = build_overall_messages(pd.Series(payload))
            raw_resp = call_openrouter(msgs, req_id, MAX_TOKENS_STAGE2)
            parsed = parse_json_response(extract_content(raw_resp))
        except Exception as e:
            err = str(e)

    overall_rows.append({
        "generated_question_id": row.get("generated_question_id", ""),
        "candidate_model": row.get("candidate_model", ""),
        "judge_model": row.get("judge_model", ""),
        "overall_label": parsed.get("overall_label", ""),
        "overall_summary": parsed.get("overall_summary", ""),
        "openrouter_request_id": req_id,
        "error_reason": err,
    })

stage2_overall_df = pd.DataFrame(overall_rows)
print("Stage 2 rows:", len(stage2_overall_df))
stage2_overall_df.head()


## 13. Saving Outputs

This block writes all final artifacts to `OUT_DIR`:
- Stage 1 metric table and raw JSONL logs;
- Stage 1 merged and Stage 1b composite tables;
- Stage 2 overall table;
- final merged table.


In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

stage1_metric_df.to_csv(STAGE1_METRIC_CSV, index=False)

with open(STAGE1_RAW_JSONL, "w", encoding="utf-8") as f:
    for r in raw_rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

stage1_merged_df.to_csv(STAGE1_MERGED_CSV, index=False)
stage1b_composite_df.to_csv(STAGE1B_COMPOSITE_CSV, index=False)
stage2_overall_df.to_csv(STAGE2_OVERALL_CSV, index=False)

final_df = stage1_merged_df.merge(
    stage2_overall_df,
    on=["generated_question_id", "candidate_model", "judge_model"],
    how="left",
    suffixes=("", "_overall")
)
final_df.to_csv(FINAL_MERGED_CSV, index=False)

print("Saved:")
print("-", STAGE1_METRIC_CSV)
print("-", STAGE1_RAW_JSONL)
print("-", STAGE1_MERGED_CSV)
print("-", STAGE1B_COMPOSITE_CSV)
print("-", STAGE2_OVERALL_CSV)
print("-", FINAL_MERGED_CSV)


## 14. Quick Diagnostics / Sanity Checks

Final checks verify required columns and provide quick aggregate score views by model.
Use this section to confirm that exported tables are structurally complete before downstream analysis.


In [ ]:
print("Missing column diagnostics")
for name, df_, required in [
    ("rules_table", rules_table, ["parent_rule_id","parent_rule_text","rule_id","rule_text_atomic","rule_figure_caption","ambiguity_remark"]),
    ("calibration_examples", calibration_examples, ["reason_gt_id","parent_rule_id","question_id","scene_id","task_family","ground_truth_answer","answer_to_evaluate"]),
    ("metric_rubrics", metric_rubrics, ["metric_name","definition","score_scale","scoring_guidelines","anti_bias_note"]),
    ("evaluation_items", evaluation_items, ["generated_question_id","template_id","question_text","scene_id","RuleID","ParentRuleID","model_answer"]),
]:
    miss = [c for c in required if c not in df_.columns]
    print(f"- {name}: missing={miss}")

if not stage1_metric_df.empty:
    print("\nAverage retrieved examples by level:")
    display(stage1_metric_df[["retrieval_level1_n","retrieval_level2_n","retrieval_level3_n","retrieval_level4_n"]].mean().to_frame("mean"))

    print("\nLocal calibration sufficiency:")
    display(stage1_metric_df["local_pool_sufficient"].value_counts(dropna=False).to_frame("count"))

    print("\nFallback usage fraction:")
    fallback_fraction = stage1_metric_df["used_fallback"].mean()
    print(float(fallback_fraction))

    print("\nAmbiguous items count:")
    print(int(stage1_metric_df["ambiguity_flag"].fillna(False).astype(bool).sum()))

score_cols = [
    c for c in final_df.columns
    if any(k in metric_key(c) for k in [
        "hallucination", "redundancy", "commonsense", "completeness", "entity grounding", "visual alignment", "crcs_deterministic", "rsf_deterministic"
    ])
]
if score_cols:
    print("\nAverage scores per model:")
    display(final_df.groupby("candidate_model")[score_cols].mean(numeric_only=True))

print("\nDiagnostics complete.")
